<div align="right"><a href="L14-Quantisierung-der-Filterkoeffizienten.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). Please note that for running MATLAB in Notebooks, you have to install the [MATLAB API for Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) first. The installation process is described [here](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/S14-Quantization-of-filter-coefficients.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S13 Designing recursive filters](./S13-Designing-recursive-filters.ipynb) | [Exercise 14](./E14-Quantization-of-filter-coefficients.ipynb) | [Contents / Overview](./00-MATLAB-Notebooks-Overview-EN.ipynb) | Next: [S15 Window Functions](./S15-Window-Functions.ipynb)

</div>

---


# Quantization of Filter Coefficients

In section 4.4.1 it was shown during the analysis of parameter quantization that the poles are more sensitive as their number increases and the distance between them decreases. For zeros, a similar behavior can be seen. Therefore, the **third canonical form** (exercise 4.4.1) is of great importance for the implementation of higher-order systems.

This exercise shows an example of how **quantization of filter coefficients** influences the system function $H(z)$. The function `lfxquant()` implements a linear quantizer:

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1} ,
$$

where $l$ is the number of bits and $v$ is the number to be quantized. To quantize the coefficients of a digital filter, it is necessary to normalize the coefficients. The function `[aq, n] = lcoefrnd(a, l)` calculates the vector of quantized coefficients `aq` as well as the normalization factor `n`.

---

### a)

For our analysis, we will use a **Cauer low-pass filter** of 7th order with the following specifications:

* $3 ,\text{dB}$ cut-off frequency: $0.4 \pi$
* Pass-band attenuation: $R_P = 0.1 ,\text{dB}$
* Stop-band attenuation: $R_S = 40 ,\text{dB}$

MATLAB’s command

```matlab
[b, a] = ellip(7, 0.1, 40, 0.4)
```

calculates the coefficients. Plot the transfer function using `freqz()`.

---

### b)

Quantize the filter coefficients with **8 bits**, i.e.:

```matlab
[bq, nb] = lcoefrnd(b, 8)
[aq, na] = lcoefrnd(a, 8)
```

to get the quantized coefficients `bq` and `aq` together with the normalization factors `nb` and `na`.

Plot the resulting transfer functions as well as the version with non-quantized coefficients (exercise 4.4.1.a).

**Hint:** To plot the transfer function of the filter with quantized parameters, the normalization of coefficients has to be inverted:

```matlab
Hq = nb/na * freqz(bq, aq, 512)
```

---

### c)

Applying **8-bit quantization**, there is a high deviation of the transfer function compared to the given tolerance scheme. The tolerable deviations are:

* $R_D^+ = 0.15 ,\text{dB}$
* $R_D^- = -0.15 ,\text{dB}$
* $R_S = -38 ,\text{dB}$

> $R_D^+$ and $R_D^-$ denote the maximal variations in the pass-band,
> where $R_D^- = R_P$.

Determine the **minimum word length** of the filter coefficients to accomplish this tolerance scheme.

---

There are different structures of digital filters. The **direct form** is implemented in MATLAB’s `filter()` function, which has been used earlier.

To reduce the effects of quantization, the **third canonical form** is mostly used, whereby the system is realized with a number of first- and second-order filters connected in series.

In the following, we analyze the effects of parameter quantization when using the **third canonical form** for the upper Cauer low-pass filter.

---

### d)

Calculate the filter coefficients of the **second-order sections (SOS)** of the Cauer low-pass using `lcascade()` and the resulting transfer function of the system:

$$
H(z) = \prod_{i=1}^{p} H_i(z)
$$

Quantize the filter coefficients of the SOS using `lcoefrnd()` with a word length of $l = 8, 10, 12, \dots$ bits, and calculate the transfer function of the cascade with quantized coefficients.

Determine how many bits are necessary to quantize the filter coefficients of the SOS to fit the tolerance scheme (exercise 4.4.1.a). Compare this length with the minimum length that was necessary for the **direct implementation** of the filter to fit the tolerance scheme (exercise 4.4.1.c).

In [ ]:
# BEGIN SOLUTION
% ##  Required m-File(s):  lfxquant.m, lcoefrnd.m, lcascade.m                ##

% #####  Task a:  Cauer Lowpass  #####
[b, a] = ellip(7, 0.1, 40, 0.4);
[H, W] = freqz(b, a, 512);

% # Graphical Output # %
figure; 
plot(W/pi, 20*log10(abs(H))); 
grid; 
xlabel('Omega/pi (Nyquist = 1)'); 
ylabel('|H(exp(j*Omega))| in dB'); 
title('Cauer Lowpass');

% #####  Task b:  Quantize Filter Coefficients  #####
[b, a] = ellip(7, 0.1, 40, 0.4);
[H, W] = freqz(b, a, 512);
[bq, nb] = scoefrnd(b, 8);
[aq, na] = scoefrnd(a, 8);
Hq = nb/na * freqz(bq, aq, 512);

% # Graphical Output # %
figure; 
plot(W/pi, 20*log10(abs(H))); 
hold on; 
plot(W/pi, 20*log10(abs(Hq)), '-.'); 
hold off; 
grid; 
title('- Unquantized Coefficients,  -. 8 Bit Quantized Coefficients'); 
xlabel('Omega/pi (Nyquist = 1)');
ylabel('|H(exp(j*Omega))| in dB');

% #####  Task c:  Determine Minimum Word Length  #####
[b, a] = ellip(7, 0.1, 40, 0.4);   % Design Filter
[H, W] = freqz(b, a, 512);         % Unquantized Transfer Function
v_length = [8, 12, 16];

for l = v_length
    [bq, nb] = scoefrnd(b, l);     % Quantize Coefficients to l Bits
    [aq, na] = scoefrnd(a, l);
    Hq = nb/na * freqz(bq, aq, 512); % Quantized Transfer Function
    
    % # Graphical Output # %
    % Passband
    figure;                      
    plot(W(1:208)/pi, 20*log10(abs(H(1:208)))); 
    axis([0 0.5 -1.4 0.3]); 
    grid; 
    hold on;
    plot(W(1:208)/pi, -0.15*ones(size(Hq(1:208))), 'w' ); % Display Tolerance Limits
    plot(W(1:208)/pi, +0.15*ones(size(Hq(1:208))), 'w' );
    plot(W(1:208)/pi, 20*log10(abs(Hq(1:208))), '--'); 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)');
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('Passband: - Unquantized,  -- Quantized with %d Bits', l);
    title(string);
    
    % Stopband
    figure;                      
    plot(W(215:512)/pi, 20*log10(abs(H(215:512)))); 
    hold on;
    plot(W(215:512)/pi, 20*log10(abs(Hq(215:512))), '--');
    plot(W(215:512)/pi, -38*ones(size(Hq(215:512))), 'w' ); 
    grid;
    hold off; 
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('Stopband: - Unquantized,  -- Quantized with %d Bits', l);
    title(string);
end


% #####  Task d:  Quantize Filter Coefficients  #####
[b, a] = ellip(7, 0.1, 40, 0.4);  % Design filter
[B, A] = scascade(b, a);          % Calculate unquantized transfer function in
                                   % 3rd canonical form
[m, n] = size(B);                 % Product of transfer functions of subsystems
Hi = zeros(512, m);

for k = 1:m
    Hi(:, k) = freqz(B(k, :), A(k, :), 512);
end
H3k = Hi(:, 1) .* Hi(:, 2) .* Hi(:, 3) .* Hi(:, 4);
Bq = zeros(size(B));              % Initialize vectors
Aq = Bq;
nb = zeros(m, 1);
na = nb;
v_length = [12];

for l = v_length

    for k = 1:m     % Quantize coefficients of subsystems to l bits
        [Bq(k, :), nb(k)] = scoefrnd(B(k, :), l);
        [Aq(k, :), na(k)] = scoefrnd(A(k, :), l);
    end
    Hq = zeros(512, m);            % Quantized overall transfer function

    for k = 1:m
        Hq(:, k) = nb(k) / na(k) * freqz(Bq(k, :), Aq(k, :), 512);
    end
    Hqg = Hq(:, 1) .* Hq(:, 2) .* Hq(:, 3) .* Hq(:, 4);

    % # Graphical Output # %
    % Passband
    figure;                     
    plot(W(1:208)/pi, 20*log10(abs(H3k(1:208))));
    axis([0 0.5 -1.4 0.3]); 
    grid; 
    hold on;
    % Display tolerance limits
    plot(W(1:208)/pi, -0.15*ones(size(Hqg(1:208))), 'w' ); 
    plot(W(1:208)/pi, +0.15*ones(size(Hqg(1:208))), 'w' );
    plot(W(1:208)/pi, 20*log10(abs(Hqg(1:208))), '--'); 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('3rd Canonical: Pass.: - Unquantized,  -- Quantized with %d Bits', l);
    title(string);
    
    % Stopband
    figure;                     
    plot(W(215:512)/pi, 20*log10(abs(H3k(215:512)))); 
    hold on;
    plot(W(215:512)/pi, 20*log10(abs(Hqg(215:512))), '--');
    plot(W(215:512)/pi, -38*ones(size(Hqg(215:512))), 'w' ); 
    grid; 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('3rd Canonical: Stop.: - Unquantized,  -- Quantized with %d Bits', l);
    title(string);
end
# END SOLUTION

## Solution: Quantization of Filter Coefficients

### a)-b)

The figure shows the **amplitude response** of the Cauer low-pass filter.

<img src="./pngs/ldsv4a04_ab.png" width="250">

If the coefficients of both polynomials (numerator and denominator) are quantized according to

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1}
$$

with 8 bits, we obtain the illustrated amplitude response. A comparison between the two functions shows a clear variation in the transition from **pass-band** to **stop-band**. The given tolerance scheme is **not fulfilled** by the system with quantized coefficients.

---

### c)

Using the tolerance variations $R_D^+$ and $R_D^-$ given in the exercise, we need at least **16-bit words** for realization in the **direct form** as a single system.

The following figures show the stop- and pass-band of the quantized transfer function in comparison with the non-quantized version:

<img src="./pngs/ldsv4a04_c1.png" width="250">
<img src="./pngs/ldsv4a04_c2.png" width="250">

For a word length of 16 bits, there is a **small variance** before the trailing edge in the pass-band, while in the stop-band, there is no variation.

---

### d)

If we realize the Cauer low-pass (exercise part a) using the **third canonical form** and determine the **minimum quantized word length** needed to fulfill the tolerance scheme, it shows that a **12-bit quantization** is enough.

Compared to the direct implementation (exercise part b), this requires **4 bits less**. The figures show the amplitude responses in the **stop-** and **pass-band**:

<img src="./pngs/ldsv4a04_d1.png" width="250">
<img src="./pngs/ldsv4a04_d2.png" width="250">